# Notebook 01 — Data prep & feature engineering

**Purpose:** Build a Tableau-ready dataset from the Stack Overflow 2025 Developer Survey.

**Inputs (local):**
- `data/raw/survey_results_public.csv` (full dataset, not committed)
- (Optional) `data/sample/survey_results_public_sample.csv` for quick preview

**Key steps:**
- Pay normalization (winsorization + log pay)
- Cohort bucketing: role, experience, remote work, org size
- Skill breadth features: languages / DBs / cloud platforms
- AI_IntensityScore + HighAI segmentation (top ~20%)

**Output (generated locally):**
- `data/processed/so2025_tableau_clean_final.csv` (Tableau-ready extract)


In [1]:
# KDD: Data Selection + Cleaning/Pre-processing for SO 2025 survey (viz-ready)
# Author: Shreeyash Baranwal
# Goal: Produce engineered fields and a clean CSV for Tableau dashboards

import pandas as pd
import numpy as np
import re
import warnings

pd.set_option("display.max_columns", 200)
warnings.filterwarnings("ignore")


In [2]:
# === CONFIG ===
RAW_PATH = "survey_results_public.csv"   # <-- change if needed
OUT_PATH = "so2025_kdd_for_tableau.csv"  # final output for Tableau

# FX/PPP conversion is OFF for now (we'll filter to USD rows for pay visuals)
ENABLE_CURRENCY_CONVERSION = False  # set True later if you add a FX map


In [3]:
# === LOAD & PRESENCE PROFILE (expanded for all hypotheses) ===
df = pd.read_csv(RAW_PATH, low_memory=False)
print("Rows, Cols:", df.shape)

# Column families (prefix-aware for matrix/ranking blocks)
FAMILIES = {
    # IDs / trace
    "ids": ["ResponseId"],
    "trace": ["Currency", "CompTotal", "MainBranch", "EmploymentAddl", "SOAccount", "SODuration", "SOComm"],

    # Compensation
    "comp": ["ConvertedCompYearly"],

    # AI adoption / usage
    "ai_use": ["AISelect", "LearnCodeAI", "AIAgents", "AIModels", "DevEnvs"],  # note: AIModels/DevEnvs are multiselect/text

    # Skills (mechanisms)
    "skills_lang": ["LanguageHaveWorkedWith", "LanguagesHaveEntry"],
    "skills_db": ["DatabaseHaveWorkedWith", "DatabaseHaveEntry"],
    "skills_cloud": ["PlatformHaveWorkedWith", "PlatformHaveEntry"],
    "skills_web": ["WebframeHaveWorkedWith", "WebframeHaveEntry"],
    "skills_ide": ["DevEnvs", "DevEnvHaveEntry"],

    # Cohorts / controls
    "cohorts": ["DevType", "OrgSize", "RemoteWork", "EdLevel", "Country", "Industry", "WorkExp", "YearsCode", "ICorPM", "Employment"],

    # Productivity proxies
    "prod": ["ToolCountWork", "ToolCountPersonal", "JobSat", "SOVisitFreq", "SOPartFreq"],
    "so_actions": ["SO_Actions"],  # prefix family SO_Actions_*

    # Attitudes / risk / friction
    "attitudes": ["AISent", "AIAcc", "AIComplex", "AIThreat", "NewRole", "SOFriction"],

    # Enablement / barriers / orchestration ecosystems
    "enable_purchase": ["PurchaseInfluence"],
    "enable_rank_endorse": ["TechEndorse"],  # TechEndorse_* (ranking)
    "enable_rank_oppose": ["TechOppose"],    # TechOppose_* (ranking)
    "agent_ecosystem": ["AIAgentKnowledge", "AIAgentOrchestration", "AIAgentObserveSecure", "AIAgentExternal",
                        "AIAgentKnowWrite", "AIAgentOrchWrite", "AIAgentObsWrite"],
    "agent_impact": ["AIAgentImpact", "AIAgentChallenges"],

    # Misc behaviors / comms that can proxy maturity
    "collab_docs": ["OfficeStackAsync", "OfficeStackHaveEntry", "OfficeStackWantEntry"],
    "comms": ["CommPlatform", "CommPlatformHaveEntr", "CommPlatformWantEntr"]
}

def has_any(prefix_or_exact):
    """True if any column equals or startswith the given string."""
    return any(c == prefix_or_exact or c.startswith(prefix_or_exact + "_") for c in df.columns)

presence_report = {}
for fam, keys in FAMILIES.items():
    fam_presence = {}
    for k in keys:
        fam_presence[k] = has_any(k)
    presence_report[fam] = fam_presence

# Pretty print a compact presence summary
missing_critical = []
print("\n=== Presence by family (True means available in this file) ===")
for fam, pres in presence_report.items():
    flags = ", ".join(f"{k}:{'✓' if v else '—'}" for k, v in pres.items())
    print(f"{fam:20s} -> {flags}")
    # flag must-haves for causal/heterogeneity
    if fam in ["ids","comp","cohorts","ai_use"] and not all(pres.values()):
        # collect missing keys
        for k, v in pres.items():
            if not v:
                missing_critical.append((fam, k))

if missing_critical:
    print("\nWARNING: Some critical fields are missing for ATE/CATE controls:")
    for fam, k in missing_critical:
        print(f"  - {fam} → {k}")
else:
    print("\nAll critical families (ids/comp/cohorts/ai_use) have at least one matching column.")


Rows, Cols: (49123, 170)

=== Presence by family (True means available in this file) ===
ids                  -> ResponseId:✓
trace                -> Currency:✓, CompTotal:✓, MainBranch:✓, EmploymentAddl:✓, SOAccount:✓, SODuration:✓, SOComm:✓
comp                 -> ConvertedCompYearly:✓
ai_use               -> AISelect:✓, LearnCodeAI:✓, AIAgents:✓, AIModels:—, DevEnvs:—
skills_lang          -> LanguageHaveWorkedWith:✓, LanguagesHaveEntry:✓
skills_db            -> DatabaseHaveWorkedWith:✓, DatabaseHaveEntry:✓
skills_cloud         -> PlatformHaveWorkedWith:✓, PlatformHaveEntry:✓
skills_web           -> WebframeHaveWorkedWith:✓, WebframeHaveEntry:✓
skills_ide           -> DevEnvs:—, DevEnvHaveEntry:✓
cohorts              -> DevType:✓, OrgSize:✓, RemoteWork:✓, EdLevel:✓, Country:✓, Industry:✓, WorkExp:✓, YearsCode:✓, ICorPM:✓, Employment:✓
prod                 -> ToolCountWork:✓, ToolCountPersonal:✓, JobSat:✓, SOVisitFreq:✓, SOPartFreq:✓
so_actions           -> SO_Actions:✓
attitudes     

In [4]:
CORE_PREVIEW = [c for c in [
    "ResponseId","Country","ConvertedCompYearly",
    "DevType","OrgSize","RemoteWork","EdLevel","Industry","WorkExp","YearsCode","ICorPM","Employment",
    "AISelect","LearnCodeAI","AIAgents","AISent","AIAcc","AIComplex","AIThreat","NewRole",
    "ToolCountWork","ToolCountPersonal","JobSat","SOVisitFreq","SOPartFreq"
] if c in df.columns]
display(df[CORE_PREVIEW].head(5))


,ResponseId,Country,ConvertedCompYearly,DevType,OrgSize,RemoteWork,EdLevel,Industry,WorkExp,YearsCode,ICorPM,Employment,AISelect,LearnCodeAI,AIAgents,AISent,AIAcc,AIComplex,AIThreat,NewRole,ToolCountWork,ToolCountPersonal,JobSat,SOVisitFreq,SOPartFreq
0,1,Ukraine,61256.0,"Developer, mobile",20 to 99 employees,Remote,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Fintech,8.0,14.0,People manager,Employed,"Yes, I use AI tools monthly or infrequently","Yes, I learned how to use AI-enabled tools for...","Yes, I use AI agents at work monthly or infreq...",Indifferent,Neither trust nor distrust,Bad at handling complex tasks,I'm not sure,I have neither consider or transitioned into a...,7.0,3.0,10.0,A few times per week,I have never participated in Q&A on Stack Over...
1,2,Netherlands,104413.0,"Developer, back-end",500 to 999 employees,"Hybrid (some in-person, leans heavy to flexibi...","Associate degree (A.A., A.S., etc.)",Retail and Consumer Services,2.0,10.0,Individual contributor,Employed,"Yes, I use AI tools weekly","Yes, I learned how to use AI-enabled tools for...","No, and I don't plan to",Indifferent,Neither trust nor distrust,Bad at handling complex tasks,I'm not sure,I have transitioned into a new career and/or i...,6.0,5.0,9.0,Multiple times per day,"Infrequently, less than once per year"
2,3,Ukraine,53061.0,"Developer, front-end",NaN,NaN,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Software Development,10.0,12.0,NaN,"Independent contractor, freelancer, or self-em...","Yes, I use AI tools daily","Yes, I learned how to use AI-enabled tools for...","Yes, I use AI agents at work weekly",Favorable,Somewhat trust,Neither good or bad at handling complex tasks,No,I have transitioned into a new career and/or i...,3.0,3.0,8.0,A few times per week,"Infrequently, less than once per year"
3,4,Ukraine,36197.0,"Developer, back-end","10,000 or more employees",Remote,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Retail and Consumer Services,4.0,5.0,Individual contributor,Employed,"Yes, I use AI tools weekly","Yes, I learned how to use AI-enabled tools for...","Yes, I use AI agents at work monthly or infreq...",Favorable,Somewhat trust,Bad at handling complex tasks,No,I have neither consider or transitioned into a...,NaN,NaN,6.0,A few times per month or weekly,I have never participated in Q&A on Stack Over...
4,5,Ukraine,60000.0,Engineering manager,NaN,NaN,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Software Development,21.0,22.0,NaN,"Independent contractor, freelancer, or self-em...","Yes, I use AI tools weekly","Yes, I learned how to use AI-enabled tools for...","No, and I don't plan to",Favorable,Neither trust nor distrust,"Good, but not great at handling complex tasks",No,I have neither consider or transitioned into a...,4.0,3.0,7.0,A few times per month or weekly,"Infrequently, less than once per year"


In [5]:
# === HELPERS: numeric coercion, winsorize, multiselect union ===
def to_numeric(s):
    return pd.to_numeric(s, errors="coerce")

def winsorize(s, p_low=0.01, p_high=0.99):
    if s is None or s.isna().all():
        return s
    lo, hi = s.quantile(p_low), s.quantile(p_high)
    return s.clip(lower=lo, upper=hi)

_SPLIT_RE = re.compile(r'\s*;\s*|\s*,\s*|\s*\|\s*')

def count_union(series_list, index=None):
    """Union-size across multiple multiselect text columns."""
    if not series_list:
        return pd.Series([np.nan] * len(index), index=index) if index is not None else pd.Series(dtype=int)
    def f_row(vals):
        bag = set()
        for v in vals:
            if isinstance(v, str) and v.strip():
                parts = [p.strip().lower() for p in _SPLIT_RE.split(v) if p and p.strip()]
                bag.update(parts)
        return len(bag)
    base = pd.DataFrame(series_list).T
    if index is not None:
        base.index = index
    return base.apply(f_row, axis=1).astype(int)


In [6]:
# === INITIALIZE OUTPUT: IDs, descriptors, experience, proxies ===
out = pd.DataFrame()
out["ResponseId"] = df.get("ResponseId", range(1, len(df)+1))

# Descriptors / filters
for col in ["Country","Currency","DevType","OrgSize","RemoteWork","EdLevel","Employment","Industry",
            "MainBranch","EmploymentAddl","SOAccount","SODuration","SOComm","ICorPM"]:
    if col in df.columns:
        out[col] = df[col]

# Experience
out["WorkExp_num"]   = to_numeric(df.get("WorkExp"))
out["YearsCode_num"] = to_numeric(df.get("YearsCode"))

# Productivity proxies / refs
# === CLEAN TOOL COUNT FIELDS ===
# Convert safely to numeric
tcw = to_numeric(df.get("ToolCountWork"))
tcp = to_numeric(df.get("ToolCountPersonal"))

# Remove impossible values (>100). True survey range is <30.
tcw = tcw.mask(tcw > 100, np.nan)
tcp = tcp.mask(tcp > 100, np.nan)

# Optional: winsorize remaining outliers (keeps long tails reasonable)
tcw = winsorize(tcw, 0.01, 0.99)
tcp = winsorize(tcp, 0.01, 0.99)

# Store back to output DataFrame
out["ToolCountWork"] = tcw
out["ToolCountPersonal"] = tcp
if "JobSat" in df.columns:
    out["JobSat"] = df["JobSat"]

# Extra behavior proxies (keep raw for later Tableau use)
for col in ["SOVisitFreq","SOPartFreq","SOFriction"]:
    if col in df.columns:
        out[col] = df[col]


In [7]:
# === COMPENSATION (USD) ===
# Local as reported
out["CompLocal"] = to_numeric(df.get("CompTotal"))

# USD (use survey-provided converted comp)
if "ConvertedCompYearly" in df.columns:
    out["Pay_USD"] = winsorize(to_numeric(df["ConvertedCompYearly"]), 0.01, 0.99)
else:
    # Worst-case fallback: USD only by currency (rare)
    out["Pay_USD"] = np.where(
        df.get("Currency","").astype(str).str.startswith("USD"),
        out["CompLocal"], np.nan
    )
    out["Pay_USD"] = winsorize(out["Pay_USD"], 0.01, 0.99)

# Log pay
out["LogPay"] = np.log(out["Pay_USD"]).replace([-np.inf, np.inf], np.nan)

print("Pay_USD non-missing:", int(out["Pay_USD"].notna().sum()))
print(out["Pay_USD"].describe(percentiles=[0.1,0.5,0.9,0.99]))


Pay_USD non-missing: 23928
count     23928.000000
mean      90879.930583
std       77541.207825
min          65.000000
10%        8712.900000
50%       75383.500000
90%      184000.000000
99%      440856.000000
max      440856.000000
Name: Pay_USD, dtype: float64


In [8]:
# === COHORT BUCKETING ===

# Role families
def bucket_role(val):
    if not isinstance(val, str):
        return np.nan
    v = val.lower()
    if any(k in v for k in ["data","ml","ai","analytics","business intelligence"]): return "Data/ML"
    if any(k in v for k in ["devops","cloud","sre","site reliability"]): return "DevOps/Cloud"
    if any(k in v for k in ["mobile"]): return "Mobile"
    if any(k in v for k in ["security","secops"]): return "Security"
    if any(k in v for k in ["db","database","dba","platform"]): return "DB/Platform"
    if any(k in v for k in ["web","front-end","frontend","full stack","backend","back-end"]): return "Web/Fullstack"
    return "Other/Unknown"

out["DevType_bucket"] = df.get("DevType").apply(bucket_role) if "DevType" in df.columns else "Other/Unknown"

# Org size ordered buckets
def map_orgsize(val):
    if not isinstance(val, str):
        return np.nan

    v = val.lower().replace(",", "")

    # If they don't know -> Unknown
    if "don't know" in v or "unknown" in v:
        return "Unknown"

    if "just me" in v or v.strip() == "1":
        return "1"

    nums = re.findall(r'\d+', v)
    if not nums:
        return "Unknown"

    n = int(nums[0])
    if n < 10: return "2–9"
    if n < 50: return "10–49"
    if n < 250: return "50–249"
    if n < 1000: return "250–999"
    if n < 5000: return "1k–4.9k"
    if n < 10000: return "5k–9.9k"
    return "10k+"

out["OrgSize_clean"] = df.get("OrgSize").apply(map_orgsize) if "OrgSize" in df.columns else np.nan

# Remote normalization
def map_remote(val):
    if not isinstance(val, str):
        return np.nan
    v = val.lower()

    # Clear Remote
    if "remote" in v:
        return "Remote"

    # Hybrid includes "hybrid" or "your choice" flexible arrangements
    if "hybrid" in v or "your choice" in v or "flexible" in v:
        return "Hybrid"

    # Remaining in-person variants
    if "in-person" in v or "office" in v or "in office" in v:
        return "Onsite"

    return np.nan   # unknown or messy responses

out["RemoteWork_clean"] = df.get("RemoteWork").apply(map_remote) if "RemoteWork" in df.columns else np.nan

# Experience band (fallback to YearsCode when WorkExp missing)
def band_exp(x):
    x = pd.to_numeric(x, errors="coerce")
    if pd.isna(x): return np.nan
    if x <= 2: return "0–2"
    if x <= 5: return "3–5"
    if x <= 10: return "6–10"
    if x <= 15: return "11–15"
    return "16+"

src_exp = out["WorkExp_num"].fillna(out["YearsCode_num"])
out["ExpBand"] = src_exp.apply(band_exp)

out[["DevType_bucket","OrgSize_clean","RemoteWork_clean","ExpBand"]].head(5)


,DevType_bucket,OrgSize_clean,RemoteWork_clean,ExpBand
0,Mobile,10–49,Remote,6–10
1,Web/Fullstack,250–999,Hybrid,0–2
2,Web/Fullstack,NaN,NaN,6–10
3,Web/Fullstack,10k+,Remote,3–5
4,Other/Unknown,NaN,NaN,16+


In [9]:
# === SKILL BREADTH (multiselect unions) ===
lang_have  = df.get("LanguageHaveWorkedWith")
lang_other = df.get("LanguagesHaveEntry")
db_have    = df.get("DatabaseHaveWorkedWith")
db_other   = df.get("DatabaseHaveEntry")
plat_have  = df.get("PlatformHaveWorkedWith")
plat_other = df.get("PlatformHaveEntry")
web_have   = df.get("WebframeHaveWorkedWith")
web_other  = df.get("WebframeHaveEntry")

out["NumLangs"]  = count_union([s for s in [lang_have, lang_other] if s is not None], index=df.index)  \
                   if ("LanguageHaveWorkedWith" in df.columns or "LanguagesHaveEntry" in df.columns) else np.nan
out["NumDBs"]    = count_union([s for s in [db_have, db_other]     if s is not None], index=df.index)  \
                   if ("DatabaseHaveWorkedWith" in df.columns or "DatabaseHaveEntry" in df.columns) else np.nan
out["NumClouds"] = count_union([s for s in [plat_have, plat_other] if s is not None], index=df.index)  \
                   if ("PlatformHaveWorkedWith" in df.columns or "PlatformHaveEntry" in df.columns) else np.nan
out["NumWeb"]    = count_union([s for s in [web_have, web_other]   if s is not None], index=df.index)  \
                   if ("WebframeHaveWorkedWith" in df.columns or "WebframeHaveEntry" in df.columns) else np.nan

for col in ["NumLangs","NumDBs","NumClouds","NumWeb"]:
    if col in out.columns and out[col].notna().any():
        print(col, "nonzero %:", float((out[col] > 0).mean()))


NumLangs nonzero %: 0.6456038922704233
NumDBs nonzero %: 0.5257211489526291
NumClouds nonzero %: 0.49659019196710297
NumWeb nonzero %: 0.47985668627730393


In [10]:
# === FLAGS: SQL_any, BI_any ===
# SQL from language/database fields
lang_all = (df.get("LanguageHaveWorkedWith", pd.Series([""]*len(df))).fillna("").astype(str)
            + ";" + df.get("LanguagesHaveEntry", pd.Series([""]*len(df))).fillna("").astype(str))
db_all   = (df.get("DatabaseHaveWorkedWith", pd.Series([""]*len(df))).fillna("").astype(str)
            + ";" + df.get("DatabaseHaveEntry", pd.Series([""]*len(df))).fillna("").astype(str))
sql_pat = re.compile(r'\bsql\b|postgres|mysql|sqlite|mssql|tsql|oracle', flags=re.I)
out["SQL_any"] = (lang_all.str.contains(sql_pat) | db_all.str.contains(sql_pat)).astype(int)

# BI_any from collaboration/dev envs/comms text families
bi_sources = []
for col in ["OfficeStackAsync","OfficeStackHaveEntry","OfficeStackWantEntry",
            "DevEnvs","DevEnvHaveEntry","DevEnvWantEntry",
            "CommPlatform","CommPlatformHaveEntr","CommPlatformWantEntr"]:
    if col in df.columns:
        bi_sources.append(df[col].fillna("").astype(str))

bi_all = pd.Series([""]*len(df), index=df.index)
for s in bi_sources:
    bi_all = bi_all.str.cat(s, sep=";")

bi_pat = re.compile(r'\bpower\s*bi\b|tableau|looker|data\s*studio|qlik', flags=re.I)
out["BI_any"] = bi_all.str.contains(bi_pat).astype(int)

print("SQL_any prevalence:", float(out["SQL_any"].mean()))
print("BI_any prevalence:", float(out["BI_any"].mean()))


SQL_any prevalence: 0.5153594039451987
BI_any prevalence: 0.00016285650306373795


In [11]:
# === AI INTENSITY (exact-label mapping) & HIGH-AI (rank-based top 20%) ===
import numpy as np
import pandas as pd

# Exact string → weight mappings (from the uniques you provided)
map_AISelect = {
    "Yes, I use AI tools daily": 2.0,
    "Yes, I use AI tools weekly": 1.6,
    "Yes, I use AI tools monthly or infrequently": 1.0,
    "No, but I plan to soon": 0.6,
    "No, and I don't plan to": -0.5,
    "NA": 0.0,
}
map_LearnCodeAI = {
    "Yes, I learned how to use AI-enabled tools required for my job or to benefit my career": 1.2,
    "Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies": 1.0,
    "No, I didn't spend time learning in the past year": -0.2,
    "No, I learned something that was not related to AI or AI enablement for my personal curiosity and/or hobbies": 0.0,
    "No, I learned something that was not related to AI or AI enablement as required for my job or to benefit my career": 0.0,
    "NA": 0.0,
}
map_AIAgents = {
    "Yes, I use AI agents at work daily": 1.5,
    "Yes, I use AI agents at work weekly": 1.2,
    "Yes, I use AI agents at work monthly or infrequently": 0.8,
    "No, but I plan to": 0.4,
    "No, and I don't plan to": -0.4,
    "No, I use AI exclusively in copilot/autocomplete mode": 0.0,
    "NA": 0.0,
}
map_AIModelsChoice = {
    "Yes": 0.5,
    "No": 0.0,
    "NA": 0.0,
}
map_DevEnvsChoice = {
    "Yes": 0.3,
    "No": 0.0,
    "NA": 0.0,
}

def map_series(col, mapping):
    s = df.get(col)
    if s is None:
        return pd.Series([0.0]*len(df), index=df.index)
    return s.map(mapping).fillna(0.0).astype(float)

# Build component scores
sc_AISelect   = map_series("AISelect",       map_AISelect)
sc_Learn      = map_series("LearnCodeAI",    map_LearnCodeAI)
sc_Agents     = map_series("AIAgents",       map_AIAgents)
sc_Models     = map_series("AIModelsChoice", map_AIModelsChoice)
sc_DevEnvs    = map_series("DevEnvsChoice",  map_DevEnvsChoice)

# Final additive score (transparent & tunable)
out["AI_IntensityScore"] = (sc_AISelect + sc_Learn + sc_Agents + sc_Models + sc_DevEnvs)

# HighAI = top 20% by *rank* (avoids zero-threshold issues)
score = out["AI_IntensityScore"].fillna(0.0)
n = len(score)
k = int(np.ceil(0.20 * n))  # desired top 20%
ranked_idx = score.sort_values(ascending=False).index
top_idx = set(ranked_idx[:k])
out["HighAI"] = out.index.to_series().apply(lambda i: 1 if i in top_idx else 0).astype(int)

# Quick sanity prints
print("AI_IntensityScore summary:")
print(score.describe(percentiles=[0.2,0.5,0.8,0.9,0.95]))
print("Component means:",
      {"AISelect": float(sc_AISelect.mean()),
       "LearnCodeAI": float(sc_Learn.mean()),
       "AIAgents": float(sc_Agents.mean()),
       "AIModelsChoice": float(sc_Models.mean()),
       "DevEnvsChoice": float(sc_DevEnvs.mean())})
print("HighAI share (target ~0.20):", float(out["HighAI"].mean()))


AI_IntensityScore summary:
count    49123.000000
mean         2.090538
std          1.921414
min         -1.100000
20%          0.000000
50%          1.800000
80%          4.000000
90%          4.800000
95%          5.300000
max          5.500000
Name: AI_IntensityScore, dtype: float64
Component means: {'AISelect': 0.9002320705168658, 'LearnCodeAI': 0.66561488508438, 'AIAgents': 0.19513059055839424, 'AIModelsChoice': 0.1719662887038658, 'DevEnvsChoice': 0.15759420230849094}
HighAI share (target ~0.20): 0.2000081428251532


In [12]:
# === QC SNAPSHOTS ===
qc = {
    "rows": len(out),
    "Pay_USD_nonmissing": int(out["Pay_USD"].notna().sum()),
    "NumLangs_nonzero_pct": float((out["NumLangs"]>0).mean()) if "NumLangs" in out else np.nan,
    "NumDBs_nonzero_pct": float((out["NumDBs"]>0).mean()) if "NumDBs" in out else np.nan,
    "NumClouds_nonzero_pct": float((out["NumClouds"]>0).mean()) if "NumClouds" in out else np.nan,
    "AIIntensity_nonmissing_pct": float(out["AI_IntensityScore"].notna().mean())
}
print("QC:", qc)

usd = out["Pay_USD"].dropna()
if len(usd):
    print("Pay_USD (winsorized) percentiles:", usd.quantile([0.1,0.5,0.9,0.99]))


QC: {'rows': 49123, 'Pay_USD_nonmissing': 23928, 'NumLangs_nonzero_pct': 0.6456038922704233, 'NumDBs_nonzero_pct': 0.5257211489526291, 'NumClouds_nonzero_pct': 0.49659019196710297, 'AIIntensity_nonmissing_pct': 1.0}
Pay_USD (winsorized) percentiles: 0.10      8712.9
0.50     75383.5
0.90    184000.0
0.99    440856.0
Name: Pay_USD, dtype: float64


In [13]:
# === EXPANDED EXPORT SELECTION ===

# Base keep list (engineered + key raw)
KEEP = [
    # IDs & traceability
    "ResponseId","Country","Currency","CompTotal","MainBranch","EmploymentAddl","SOAccount","SODuration","SOComm",

    # Compensation
    "ConvertedCompYearly",  # raw USD
    "Pay_USD","LogPay",     # engineered

    # AI adoption / usage
    "AISelect","LearnCodeAI","AIAgents","AIModels","DevEnvs",
    "AI_IntensityScore","HighAI",

    # Skills parents (reproducibility)
    "LanguageHaveWorkedWith","LanguagesHaveEntry",
    "DatabaseHaveWorkedWith","DatabaseHaveEntry",
    "PlatformHaveWorkedWith","PlatformHaveEntry",
    "WebframeHaveWorkedWith","WebframeHaveEntry",
    "DevEnvHaveEntry",

    # Engineered skill breadth & flags
    "NumLangs","NumDBs","NumClouds","NumWeb","SQL_any","BI_any",

    # Cohorts / controls
    "DevType","DevType_bucket","OrgSize","OrgSize_clean","RemoteWork","RemoteWork_clean",
    "EdLevel","Employment","Industry","ICorPM",
    "WorkExp","WorkExp_num","YearsCode","YearsCode_num","ExpBand",

    # Productivity proxies / participation
    "ToolCountWork","ToolCountPersonal","JobSat","SOVisitFreq","SOPartFreq","SOFriction",

    # Attitudes / risk
    "AISent","AIAcc","AIComplex","AIThreat","NewRole",

    # Enablement / influence & agent ecosystems
    "PurchaseInfluence",
    "AIAgentKnowledge","AIAgentOrchestration","AIAgentObserveSecure","AIAgentExternal",
    "AIAgentKnowWrite","AIAgentOrchWrite","AIAgentObsWrite",
    "AIAgentImpact","AIAgentChallenges"
]

# Dynamically include ranking/matrix families with many columns
prefix_families = ["SO_Actions_", "TechEndorse_", "TechOppose_"]
dynamic_cols = [c for c in df.columns for p in prefix_families if c.startswith(p)]

# Merge engineered 'out' columns and original 'df' for raw parents
export_frame = out.copy()
for col in df.columns:
    if col not in export_frame.columns and (col in KEEP or any(col.startswith(p) for p in prefix_families)):
        export_frame[col] = df[col]

# Final keep list resolved against columns present
KEEP_ALL = [c for c in KEEP if c in export_frame.columns] + dynamic_cols
# de-duplicate preserve order
KEEP_ALL = list(dict.fromkeys(KEEP_ALL))

# Save
OUT_PATH = OUT_PATH if "OUT_PATH" in globals() else "so2025_kdd_for_tableau.csv"
export_frame[KEEP_ALL].to_csv(OUT_PATH, index=False)
print(f"Saved Tableau dataset to: {OUT_PATH}")
print("Exported columns:", len(KEEP_ALL))


Saved Tableau dataset to: so2025_kdd_for_tableau.csv
Exported columns: 99


In [14]:
# === MISSINGNESS AUDIT: columns & rows ===
import pandas as pd
import numpy as np

# Use the latest working frame; fall back to 'out' if needed
df_base = export_frame.copy() if 'export_frame' in globals() else out.copy()

# Column-level missingness
col_missing = df_base.isna().mean().sort_values(ascending=False).to_frame("missing_frac")
print("Top 30 most-missing columns:")
display(col_missing.head(30))

# Row-level missingness (overall)
row_missing = df_base.isna().mean(axis=1)
print("Row missingness percentiles:")
print(row_missing.quantile([0.5, 0.8, 0.9, 0.95, 0.99]))

# Keep a copy for pruning steps
df_work = df_base.copy()


Top 30 most-missing columns:


,missing_frac
AIAgentObsWrite,0.994626
AIAgentOrchWrite,0.990290
AIAgentKnowWrite,0.984427
SO_Actions_15_TEXT,0.983246
TechOppose_15_TEXT,0.966513
TechEndorse_13_TEXT,0.959123
DatabaseHaveEntry,0.956273
AIAgentObserveSecure,0.945219
DevEnvHaveEntry,0.943815
WebframeHaveEntry,0.933025


Row missingness percentiles:
0.50    0.27
0.80    0.73
0.90    0.83
0.95    0.86
0.99    0.90
dtype: float64


In [15]:
# === PRUNE: drop very sparse columns; optional sparse rows ===

# 1) Drop columns missing > 80% (safe; preserves most info)
COL_DROP_THRESH = 0.80
too_sparse_cols = col_missing[col_missing["missing_frac"] > COL_DROP_THRESH].index.tolist()

# Always keep IDs / criticals even if sparse (paranoia)
ALWAYS_KEEP = set([
    "ResponseId","Pay_USD","LogPay","AI_IntensityScore","HighAI",
    "DevType","DevType_bucket","OrgSize_clean","RemoteWork_clean","ExpBand",
    "Country","Industry","EdLevel","Employment"
])
too_sparse_cols = [c for c in too_sparse_cols if c not in ALWAYS_KEEP]

print(f"Dropping {len(too_sparse_cols)} very sparse columns (> {int(COL_DROP_THRESH*100)}% missing).")
df_work = df_work.drop(columns=too_sparse_cols, errors="ignore")

# 2) (Optional) Drop rows that are > 90% missing overall
ROW_DROP_THRESH = 0.90
row_missing_after = df_work.isna().mean(axis=1)
rows_to_drop = row_missing_after[row_missing_after > ROW_DROP_THRESH].index
print(f"Dropping {len(rows_to_drop)} rows (> {int(ROW_DROP_THRESH*100)}% missing).")
df_work = df_work.drop(index=rows_to_drop).reset_index(drop=True)

print("Shape after pruning:", df_work.shape)


Dropping 15 very sparse columns (> 80% missing).
Dropping 0 rows (> 90% missing).
Shape after pruning: (49123, 85)


In [16]:
# === NORMALIZE: MainBranch, SOAccount, SODuration, SOComm ===
import re

def normalize_strip(x):
    return x.strip() if isinstance(x, str) else x

# --- MainBranch: collapse to tidy labels + sort key
map_main = {
    "I am a developer by profession": "Pro developer",
    "I am not primarily a developer, but I write code sometimes as part of my work/studies": "Occasional coder",
    "I used to be a developer by profession, but no longer am": "Former developer",
    "I code primarily as a hobby": "Hobbyist",
    "I work with developers or my work supports developers but am not a developer by profession": "Dev-adjacent",
    "I am learning to code": "Learning to code",
}
order_main = ["Pro developer","Dev-adjacent","Occasional coder","Hobbyist","Learning to code","Former developer"]

if "MainBranch" in df_work.columns:
    df_work["MainBranch_clean"] = df_work["MainBranch"].map(map_main).fillna(df_work["MainBranch"])
    order_map = {lab:i for i, lab in enumerate(order_main, start=1)}
    df_work["MainBranch_sort"] = df_work["MainBranch_clean"].map(order_map).astype("Int64")

# --- SOAccount: normalize + binary flag for convenience
map_soacct = {
    "Yes": "Yes",
    "No": "No",
    "Not sure/can't remember": "Not sure"
}
if "SOAccount" in df_work.columns:
    df_work["SOAccount_clean"] = df_work["SOAccount"].map(map_soacct).fillna(df_work["SOAccount"])
    df_work["HasSOAccount"] = df_work["SOAccount_clean"].map({"Yes":1,"No":0}).astype("Int64")

# --- SODuration: order buckets + numeric midpoint approximation
order_dur = [
    "Less than one year",
    "Between 1 and 3 years",
    "Between 3 and 5 years",
    "Between 5 and 10 years",
    "Between 10 and 15 years",
    "More than 15 years, or since Stack Overflow started in 2008",
    "I don't use Stack Overflow"
]
midpoint_dur = {
    "Less than one year": 0.5,
    "Between 1 and 3 years": 2.0,
    "Between 3 and 5 years": 4.0,
    "Between 5 and 10 years": 7.5,
    "Between 10 and 15 years": 12.5,
    "More than 15 years, or since Stack Overflow started in 2008": 17.0,
    "I don't use Stack Overflow": np.nan
}
if "SODuration" in df_work.columns:
    df_work["SODuration_clean"] = df_work["SODuration"].apply(normalize_strip)
    df_work["SODuration_sort"] = df_work["SODuration_clean"].map({v:i for i,v in enumerate(order_dur, start=1)}).astype("Int64")
    df_work["SODuration_years"] = df_work["SODuration_clean"].map(midpoint_dur).astype(float)

# --- SOComm (community identification): map to Likert + sort
order_comm = ["No, not at all","No, not really","Neutral","Yes, somewhat","Yes, definitely","Not sure"]
likert_comm = {
    "No, not at all": 1,
    "No, not really": 2,
    "Neutral": 3,
    "Yes, somewhat": 4,
    "Yes, definitely": 5,
    "Not sure": pd.NA
}
if "SOComm" in df_work.columns:
    df_work["SOComm_clean"] = df_work["SOComm"].apply(normalize_strip)
    df_work["SOComm_likert"] = df_work["SOComm_clean"].map(likert_comm).astype("Int64")
    df_work["SOComm_sort"] = df_work["SOComm_clean"].map({v:i for i,v in enumerate(order_comm, start=1)}).astype("Int64")

print("Normalization complete. Quick checks:")
for c in ["MainBranch_clean","SOAccount_clean","SODuration_clean","SOComm_clean"]:
    if c in df_work.columns:
        print(c, "→ uniques:", df_work[c].nunique())


Normalization complete. Quick checks:
MainBranch_clean → uniques: 6
SOAccount_clean → uniques: 3
SODuration_clean → uniques: 7
SOComm_clean → uniques: 6


In [17]:
# === EMPLOYMENTAddl: multi-select fan-out into Top-K flags + count ===
import re
from collections import Counter

def split_multi(val):
    if not isinstance(val, str) or not val.strip():
        return []
    parts = re.split(r'\s*;\s*|\s*\|\s*', val.strip())   # remove comma splitting
    return [p for p in (x.strip() for x in parts) if p]

def slugify(s):
    s = s.lower()
    s = re.sub(r'[^a-z0-9]+', '_', s).strip('_')
    return s[:36]  # keep column names short

if "EmploymentAddl" in df_work.columns:
    tokens_series = df_work["EmploymentAddl"].apply(split_multi)
    # Count individual tokens (not the whole combos)
    counter = Counter(tok for lst in tokens_series for tok in lst)
    TOP_K = 12  # adjust if you want more/less flags
    top_tokens = [t for t,_ in counter.most_common(TOP_K)]
    top_slugs = [slugify(t) for t in top_tokens]
    print("Top EmploymentAddl tokens:", list(zip(top_slugs, top_tokens)))

    # Create boolean flags for top tokens
    for slug, tok in zip(top_slugs, top_tokens):
        df_work[f"EmpAddl_{slug}"] = tokens_series.apply(lambda lst: 1 if tok in lst else 0).astype("Int64")

    # Count how many additional activities each respondent reports
    df_work["EmpAddl_count"] = tokens_series.apply(len).astype("Int64")

    # An 'Other' bucket if there are non-top tokens present
    top_set = set(top_tokens)
    df_work["EmpAddl_Other"] = tokens_series.apply(lambda lst: 1 if any(t not in top_set for t in lst) else 0).astype("Int64")


Top EmploymentAddl tokens: [('none_of_the_above', 'None of the above'), ('caring_for_dependents_children_elder', 'Caring for dependents (children, elderly, etc.)'), ('engaged_in_paid_work_20_29_hours_per', 'Engaged in paid work (20-29 hours per week)'), ('volunteering_regularly', 'Volunteering (regularly)'), ('attending_school_full_time', 'Attending school (full-time)'), ('engaged_in_paid_work_less_than_10_ho', 'Engaged in paid work (less than 10 hours per week)'), ('attending_school_part_time', 'Attending school (part-time)'), ('engaged_in_paid_work_10_19_hours_per', 'Engaged in paid work (10-19 hours per week)'), ('transitioning_to_retirement_graduall', 'Transitioning to retirement (gradually reducing work hours)')]


In [18]:
# === SAVE: cleaned SLIM file for Tableau ===
# Collect only once – avoid duplicates
emp_cols = [c for c in df_work.columns if c.startswith("EmpAddl_")]

SLIM_KEEP = [
    # IDs & filters
    "ResponseId","Country","Industry","EdLevel","Employment","ICorPM",
    # Compensation
    "Pay_USD","LogPay",
    # AI adoption & intensity
    "AISelect","LearnCodeAI","AIAgents","AIModelsChoice","DevEnvsChoice",
    "AI_IntensityScore","HighAI",
    # Cohorts / controls
    "DevType","DevType_bucket","OrgSize_clean","RemoteWork_clean","ExpBand",
    "WorkExp_num","YearsCode_num",
    # Skills breadth & flags
    "NumLangs","NumDBs","NumClouds","NumWeb","SQL_any","BI_any",
    # Productivity / engagement
    "ToolCountWork","ToolCountPersonal","JobSat","SOVisitFreq","SOPartFreq",
    # Attitudes / risk
    "AISent","AIAcc","AIComplex","AIThreat","SOFriction","NewRole",
    # Normalized fields
    "MainBranch_clean","MainBranch_sort","SOAccount_clean","HasSOAccount",
    "SODuration_clean","SODuration_sort","SODuration_years",
    "SOComm_clean","SOComm_likert","SOComm_sort",
] + emp_cols   # <-- EmpAddl columns added here ONLY ONCE

# Finalize
SLIM_KEEP = [c for c in SLIM_KEEP if c in df_work.columns]
df_slim = df_work[SLIM_KEEP].copy()
df_slim.to_csv("data/processed/so2025_tableau_clean_final.csv", index=False)

SLIM_KEEP = [c for c in SLIM_KEEP if c in df_work.columns]
df_slim = df_work[SLIM_KEEP].copy()

SLIM_PATH = "so2025_tableau_slim_cleaned.csv"
df_slim.to_csv(SLIM_PATH, index=False)
print(f"Saved cleaned SLIM to: {SLIM_PATH}  | shape={df_slim.shape}")


Saved cleaned SLIM to: so2025_tableau_slim_cleaned.csv  | shape=(49123, 58)
